# 03_daily_flows.ipynb
## Агрегация дневных денежных потоков

In [ ]:
import pandas as pd
import numpy as np
import os

print("=== Cash Flow at Risk - Daily Aggregation ===\n")

In [ ]:
# Путь к очищенному файлу
CSV_CLEAN = "data/transactions_clean.csv"

# Если файла нет, создаём тестовые данные
if not os.path.exists(CSV_CLEAN):
    print("⚠️ Файл transactions_clean.csv не найден!")
    print("   Создаю тестовые данные для отладки...\n")
    
    np.random.seed(42)
    test_data = pd.DataFrame({
        'trans_date': pd.date_range('2024-01-01', periods=365, freq='D'),
        'amount': np.random.normal(1000, 500, 365),
        'trans_type': np.random.choice(['inflow', 'outflow'], 365, p=[0.6, 0.4])
    })
    # Добавляем несколько отрицательных дней для реалистичности
    test_data.loc[50:60, 'amount'] = -2000
    test_data.loc[50:60, 'trans_type'] = 'outflow'
    
    test_data.to_csv(CSV_CLEAN, index=False)
    print(f"✅ Создан тестовый файл: {CSV_CLEAN}\n")

In [ ]:
# Загружаем данные
df = pd.read_csv(CSV_CLEAN)
df['trans_date'] = pd.to_datetime(df['trans_date']).dt.date

# Агрегация по дням
daily = df.groupby('trans_date').apply(
    lambda x: pd.Series({
        'inflow': x[x['trans_type'] == 'inflow']['amount'].sum(),
        'outflow': x[x['trans_type'] == 'outflow']['amount'].sum(),
        'net_flow': x[x['trans_type'] == 'inflow']['amount'].sum() - x[x['trans_type'] == 'outflow']['amount'].sum(),
        'txn_count': len(x)
    })
).reset_index()

daily.columns = ['day', 'inflow', 'outflow', 'net_flow', 'txn_count']

# Сохраняем
daily.to_csv("daily_cashflow.csv", index=False)

print(f"📊 Результат агрегации:")
print(f"   - Период: {daily['day'].min()} → {daily['day'].max()}")
print(f"   - Всего дней: {len(daily)}")
print(f"   - Суммарный приток: {daily['inflow'].sum():,.2f}")
print(f"   - Суммарный отток: {daily['outflow'].sum():,.2f}")
print(f"   - Чистый поток: {daily['net_flow'].sum():,.2f}")

print(f"\n✅ Сохранено в daily_cashflow.csv")
print("\n📋 Первые 5 дней:")
print(daily.head())